# Test src/inference.py in Colab
A throwaway test notebook -- proves the final DPO-aligned model loads and
answers correctly via the exact same `generate_answer()` function that
ships in `src/inference.py`. Not one of the 3 required deliverable
notebooks; just a sanity check.

In [1]:
# mount Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

if not os.path.exists(f'{PROJECT}/adapters/stage3_dpo/adapter_config.json'):
    raise FileNotFoundError(
        "adapters/stage3_dpo is missing -- run notebooks/dpo_alignment.ipynb "
        "(Stage 3) all the way through first, this test needs that adapter."
    )
print('Found DPO adapter at:', f'{PROJECT}/adapters/stage3_dpo')

Mounted at /content/drive
Found DPO adapter at: /content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo


In [2]:
# fail fast if there's no GPU, with clear instructions
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this runtime.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, "
        "then Runtime -> Restart session, then run all cells again from the top."
    )
print("GPU OK:", torch.cuda.get_device_name(0))

GPU OK: Tesla T4


In [3]:
%%capture
# Cell 1 -- install Unsloth
!pip install --no-deps unsloth
!pip install unsloth_zoo
!pip install --no-deps peft accelerate bitsandbytes xformers

In [4]:
# inference.py logic
from unsloth import FastLanguageModel

MODEL_PATH = f"{PROJECT}/adapters/stage3_dpo"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=2048,
    dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def generate_answer(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("Model loaded, generate_answer() ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded, generate_answer() ready.


In [5]:
# the actual test
question = "How can I apply for a refund?"
answer = generate_answer(question)
print("Q:", question)
print("A:", answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I apply for a refund?
A: I understand your eagerness to learn how you can initiate the process of requesting a refund. To ensure a smooth and satisfactory experience, please follow these steps:

1. Contact Us: Reach out to our customer support team via phone or live chat on our website. Provide them with all the details regarding the purchase you would like to cancel, including the order number.

2. Initiate Cancellation: Once you have contacted us, they will guide you through the cancellation process. They may ask you some questions about the reason for the cancellation or provide additional information as needed.

3. Review Terms and Conditions: It's essential to review the terms and conditions associated with refunds to ensure that your request complies with our policies. If there are any specific requirements or limitations, we will inform you accordingly.

4. Obtain Refund: After completing the necessary steps, our finance department will process the refund. The exact t

In [6]:
# try a couple more, just to be thorough
for q in [
    "How do I cancel an order I just placed?",
    "The item I received doesn't match the description, what can I do?",
]:
    print("Q:", q)
    print("A:", generate_answer(q))
    print("---")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I cancel an order I just placed?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: I'm sorry to hear that you're unsure about how to cancel your recent order. Don't worry, I've got you covered! To cancel your order, please follow these simple steps:

1. Access Your Account: Log in to your account portal at support.example.com using the credentials provided during sign-up.

2. Locate Your Order History: Once logged in, navigate to the "Order History" or "Order History" section within your account dashboard.

3. Find the Specific Order: Look for the order associated with the #SC-27960 and click on it to view more details.

4. Initiate Cancellation: Within the order details page, locate the option labeled "Order History" and select it.

5. Confirm the Cancellation: The system will prompt you to confirm if you would like to proceed with the cancellation of the order. Please provide any additional information as requested.

Remember, we want to ensure that you have a seamless experience, so feel free to reach
---
Q: The item I received doesn't match the description, wh